# 📑 KIỂM TRA DỮ LIỆU VĂN BẢN PHÁP LUẬT HẢI QUAN & RAG RETRIEVAL

Notebook này cho phép bạn kiểm tra và đối chiếu dữ liệu văn bản pháp luật Hải quan được trích xuất từ các file `.doc`/`.pdf`, lưu trữ trong cơ sở dữ liệu **PostgreSQL (SQL)** và **pgvector**.

In [1]:
import sys
import os
import asyncio
from sqlalchemy import select

# Thêm thư mục gốc vào PYTHONPATH khi mở notebook từ thư mục notebooks/
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

# Nạp cấu hình ứng dụng
from app.database.connection import async_session, LegalDocument, LegalArticle, CustomsDocument
from app.services.vector_store import similarity_search_structured
from app.services.ingest import seed_all_data

print("✅ Đã kết nối thành công với Hệ thống RAG Hải quan!")

NameError: name '__file__' is not defined

## 1. Nạp Dữ Liệu Thực (Seed Data Pipeline)
Chạy lệnh nạp danh mục 11 Văn bản Pháp luật (Luật Hải quan 2014, Nghị định 08/2015, 128/2020, 169/2026, Thông tư 38/2015, 33/2023, 121/2025...) vào SQL.

In [ ]:
await seed_all_data()
print("🎉 Đã nạp thành công dữ liệu vào PostgreSQL!")

## 2. Kiểm Tra Danh Mục Văn Bản (Bảng `legal_documents`)
Xem tất cả văn bản, số hiệu, loại văn bản, ngày ban hành và trạng thái hiệu lực (Hiệu lực vs Bị thay thế).

In [ ]:
async with async_session() as session:
    stmt = select(LegalDocument)
    res = await session.execute(stmt)
    docs = res.scalars().all()
    
print(f"📌 TỔNG SỐ VĂN BẢN QUẢN LÝ: {len(docs)}\n")
print("{:<5} | {:<16} | {:<10} | {:<12} | {:<15} | {:<20}".format("ID", "Số Hiệu", "Loại", "Ngày Ban Hành", "Trạng Thái", "Văn Bản Thay Thế"))
print("-" * 90)
for d in docs:
    status_str = "✅ Còn hiệu lực" if d.status == 'con_hieu_luc' else "⚠️ BỊ THAY THẾ"
    sub_str = d.superseded_by if d.superseded_by else "-"
    print("{:<5} | {:<16} | {:<10} | {:<12} | {:<15} | {:<20}".format(d.id, d.law_number, d.doc_type, str(d.issue_date), status_str, sub_str))

## 3. Kiểm Tra Chi Tiết Điều / Khoản (Bảng `customs_docs` & `legal_articles`)
Kiểm tra nội dung đã được tách theo Điều/Khoản kèm cảnh báo pháp lý đối với văn bản đã hết hiệu lực.

In [ ]:
async with async_session() as session:
    stmt = select(CustomsDocument).limit(5)
    res = await session.execute(stmt)
    articles = res.scalars().all()

for a in articles:
    print(f"==================================================")
    print(f"📜 Số hiệu: {a.law_number} | {a.article_number}: {a.title}")
    print(f"Trạng thái: {a.status} (Thay thế bởi: {a.superseded_by})")
    print(f"Nội dung trích đoạn:\n{a.content[:300]}...\n")

## 4. Test Độc Lập Luồng RAG Retrieval (Truy Hồi Tương Đồng Vector & Citation)
Thử nghiệm gửi câu hỏi tra cứu thủ tục hải quan và kiểm tra kết quả trích dẫn trả về từ PostgreSQL / BM25 Fallback.

In [ ]:
test_queries = [
    "Hồ sơ hải quan gồm những chứng từ gì?",
    "Địa điểm làm thủ tục hải quan ở đâu?",
    "Xử phạt vi phạm hành chính khai hải quan theo quy định nào?"
]

async with async_session() as session:
    for q in test_queries:
        print(f"\n🔎 CÂU HỎI TRA CỨU: '{q}'")
        results = await similarity_search_structured(session, q, limit=2)
        for idx, r in enumerate(results, 1):
            print(f"  [Nguồn {idx}]: {r['law_number']} - {r['article_number']} ({r['title']})")
            print(f"    Trạng thái: {r['status']} | Cảnh báo: {r['superseded_by']}")
            print(f"    Trích đoạn: {r['content'][:150]}...")